SỬ DỤNG PYTHON 3.12 TRÊN KAGGLE ĐỂ DEMO

In [1]:
!pip install -q underthesea

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 56.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 54.9 MB/s eta 0:00:00


In [2]:
import os
import re
import torch
from transformers import AutoTokenizer, RobertaForSequenceClassification
from underthesea import word_tokenize

print("🚀 BẮT ĐẦU KHỞI ĐỘNG HỆ THỐNG DEMO...")

# ==========================================
# 1. NẠP TỪ ĐIỂN TEENCODE
# ==========================================
file_path = '/kaggle/input/datasets/conbobietbay/teencode-dict/acronyms_dictionary.txt'
ACRONYM_DICT = {}

print("🔍 Đang nạp từ điển teencode...")
try:
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip() # Bỏ khoảng trắng 2 đầu
            # Bỏ qua các dòng trống hoặc dòng comment bắt đầu bằng '#'
            if not line or line.startswith('#'):
                continue
            
            # Cắt dựa trên dấu '=' theo đúng format của bạn
            parts = line.split('=')
            if len(parts) >= 2:
                key = parts[0].strip().lower()
                value = parts[1].strip().lower()
                ACRONYM_DICT[key] = value
                
    print(f"✅ Quá dữ! Đã nạp thành công {len(ACRONYM_DICT)} từ lóng/viết tắt!")
except Exception as e:
    print(f"⚠️ Lỗi đọc file teencode: {e}")

# ==========================================
# 2. HÀM TIỀN XỬ LÝ DỮ LIỆU
# ==========================================
def preprocess_text(text):
    text = text.lower()
    
    # Chuyển teencode
    words = text.split()
    text = " ".join([ACRONYM_DICT.get(w, w) for w in words])
    
    # Làm sạch
    text = re.sub(r'[^\s\wáàảãạăắằẳẵặâấầẩẫậéèẻẽẹêếềểễệóòỏõọôốồổỗộơớờởỡợíìỉĩịúùủũụưứừửữựýỳỷỹỵđ_]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Tách từ (Word Segmentation)
    text = word_tokenize(text, format="text")
    
    return text

# ==========================================
# 3. NẠP MÔ HÌNH PHOBERT
# ==========================================
print("🔍 Đang tìm và nạp mô hình PhoBERT...")
sentiment_model_path = None
topic_model_path = None

for root, dirs, files in os.walk('/kaggle/input'):
    if 'config.json' in files:
        if 'sentiment_model' in root:
            sentiment_model_path = root
        elif 'topic_model' in root:
            topic_model_path = root

if not all([sentiment_model_path, topic_model_path]):
    print("❌ Lỗi: Không tìm thấy model!")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tokenizer = AutoTokenizer.from_pretrained(sentiment_model_path, local_files_only=True)
    
    model_sentiment = RobertaForSequenceClassification.from_pretrained(sentiment_model_path, local_files_only=True).to(device)
    model_sentiment.eval()
    
    model_topic = RobertaForSequenceClassification.from_pretrained(topic_model_path, local_files_only=True).to(device)
    model_topic.eval()
    print(f"✅ Lõi PhoBERT đã sẵn sàng trên: {device}")

dict_sentiment = {0: '🔴 Tiêu cực', 1: '⚪ Trung tính', 2: '🟢 Tích cực'}
dict_topic = {0: '👨‍🏫 Giảng viên', 1: '📚 Chương trình', 2: '🏫 Cơ sở vật chất', 3: '❓ Khác'}

# ==========================================
# 4. CHẠY DEMO THỰC TẾ
# ==========================================
print("\n" + "="*50)
print("🎯 CHƯƠNG TRÌNH DEMO PHOBERT (BẢN CHÍNH THỨC)")
print("Nhập 'q' hoặc 'exit' để thoát.")
print("="*50 + "\n")

while True:
    raw_text = input("✍️ Mời nhập câu nhận xét (thử dùng teencode/từ viết tắt nha): ")
    
    if raw_text.lower() in ['q', 'exit', 'thoat', 'quit']:
        print("👋 Đã thoát chương trình Demo. Chúc nhóm báo cáo thành công rực rỡ!")
        break
        
    if not raw_text.strip():
        continue

    # Tiền xử lý
    clean_text = preprocess_text(raw_text)
    
    # Dự đoán
    inputs = tokenizer(clean_text, return_tensors="pt", padding=True, truncation=True, max_length=256).to(device)
    
    with torch.no_grad():
        out_s = model_sentiment(**inputs)
        out_t = model_topic(**inputs)
        
        pred_s_idx = torch.argmax(out_s.logits, dim=1).item()
        pred_t_idx = torch.argmax(out_t.logits, dim=1).item()
        
        label_s = int(model_sentiment.config.id2label[pred_s_idx])
        label_t = int(model_topic.config.id2label[pred_t_idx])
        
    print("-" * 50)
    print(f"Câu gốc        : {raw_text}")
    print(f"Câu sau xử lý  : {clean_text}")
    print(f"Chủ đề         : {dict_topic[label_t]}")
    print(f"Cảm xúc        : {dict_sentiment[label_s]}")
    print("-" * 50 + "\n")

🚀 BẮT ĐẦU KHỞI ĐỘNG HỆ THỐNG DEMO...
🔍 Đang nạp từ điển teencode...
✅ Quá dữ! Đã nạp thành công 579 từ lóng/viết tắt!
🔍 Đang tìm và nạp mô hình PhoBERT...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ Lõi PhoBERT đã sẵn sàng trên: cpu

🎯 CHƯƠNG TRÌNH DEMO PHOBERT (BẢN CHÍNH THỨC)
Nhập 'q' hoặc 'exit' để thoát.



✍️ Mời nhập câu nhận xét (thử dùng teencode/từ viết tắt nha):  gv dạy nhiệt tình, bài giảng rấc dể hỉu, 10 điểm k có nhưng!


...

--------------------------------------------------
Câu gốc        : Thầy cho bài tập về nhà dễ lắm, làm từ 8h tối đến 3h sáng mới xong 1 câu.
Câu sau xử lý  : thầy cho bài_tập về nhà dễ lắm làm từ 8 h tối đến 3 h sáng mới xong 1 câu
Chủ đề         : 👨‍🏫 Giảng viên
Cảm xúc        : 🔴 Tiêu cực
--------------------------------------------------



✍️ Mời nhập câu nhận xét (thử dùng teencode/từ viết tắt nha):  máy tính phòng thực hành chạy code nhanh như rùa bò, nhấn run xong đi ún li cafe dề vẫn chư xong.


--------------------------------------------------
Câu gốc        : máy tính phòng thực hành chạy code nhanh như rùa bò, nhấn run xong đi ún li cafe dề vẫn chư xong.
Câu sau xử lý  : máy_tính phòng thực_hành chạy code nhanh như rùa bò nhấn run xong đi ún_li cafe dề vẫn chư xong
Chủ đề         : 🏫 Cơ sở vật chất
Cảm xúc        : 🔴 Tiêu cực
--------------------------------------------------



✍️ Mời nhập câu nhận xét (thử dùng teencode/từ viết tắt nha):  q


👋 Đã thoát chương trình Demo. Chúc nhóm báo cáo thành công rực rỡ!
